In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import pathlib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

def scaledLog(col):
    return scaler.fit_transform(np.log1p(col).values.reshape(-1, 1))

def scaled(col):
    return scaler.fit_transform(col.values.reshape(-1, 1))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
filepath = '/kaggle/input/spaceship-titanic/train.csv'
df = pd.read_csv(filepath)
tst_df = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
tst_path='/kaggle/input/spaceship-titanic/test.csv'

tst_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


# Data Cleaning

In [3]:
categorical_values =df.select_dtypes(include=['object']).columns
numerical_values = df.select_dtypes(exclude=['object']).columns

#replacing null values

for cat_col in categorical_values:
    df[cat_col].fillna(df[cat_col].mode()[0], inplace=True)

for num_col in numerical_values:
    df[num_col].fillna(df[num_col].mean(), inplace=True)

len(df)

/tmp/ipykernel_17/43333184.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[cat_col].fillna(df[cat_col].mode()[0], inplace=True)
/tmp/ipykernel_17/43333184.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cat_col].fillna(df[cat_col].mode()[0], inplace=True)
/tmp/ipykernel_17/43333184.py:10: FutureWa

8693

In [4]:
categorical_variables =tst_df.select_dtypes(include=['object']).columns
numerical_variables = tst_df.select_dtypes(exclude=['object']).columns

#replacing null values

for cat_col in categorical_variables:
    tst_df[cat_col].fillna(tst_df[cat_col].mode()[0], inplace=True)

for num_col in numerical_variables:
    tst_df[num_col].fillna(tst_df[num_col].mean(), inplace=True)

len(tst_df)

/tmp/ipykernel_17/1088328831.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  tst_df[cat_col].fillna(tst_df[cat_col].mode()[0], inplace=True)
/tmp/ipykernel_17/1088328831.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tst_df[cat_col].fillna(tst_df[cat_col].mode()[0], inplace=True)
/tmp/ipykernel_17/1088

4277

# Analysis

In [5]:
#Droping the name column
df.drop(['Name'], axis=1, inplace=True)

#building the deck and port features from the 'cabin' feature
df['Deck'] = df['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'M')
df['Port'] = df['Cabin'].apply(lambda s: s[-1] if pd.notnull(s) else 'M')
df['Deck'] = df["Deck"].map({'B':0, 'F':1, 'A':2, 'G':3, 'E':4, 'D':5, 'C':6, 'T':7, 'M':8})
df['Port'] = df["Port"].map({'P':0, 'S':1}).astype(int)
df.drop(['Cabin'], axis=1, inplace=True)


df['HomePlanet']= df['HomePlanet'].map({'Earth':0, 'Europa':1,'Mars':2}).astype(int)

unique_destinations = df["Destination"].unique()
df["Destination"] = df["Destination"].map(dict(zip(unique_destinations, list(range(len(unique_destinations)))))).astype(int)
df["Destination"].unique()

if 'train' in filepath:
    df.drop(['PassengerId'],axis=1,inplace=True)

unique_cryosleep=df["CryoSleep"].unique()
df["CryoSleep"]=df["CryoSleep"].map(dict(zip(unique_cryosleep, list(range(len(unique_cryosleep)))))).astype(int)

unique_VIP=df["VIP"].unique()
df["VIP"]=df["VIP"].map(dict(zip(unique_VIP, list(range(len(unique_VIP)))))).astype(int)

df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Port
0,1,0,0,39.0,0,0.0,0.0,0.0,0.0,0.0,False,0,0
1,0,0,0,24.0,0,109.0,9.0,25.0,549.0,44.0,True,1,1
2,1,0,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,False,2,1
3,1,0,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,False,2,1
4,0,0,0,16.0,0,303.0,70.0,151.0,565.0,2.0,True,1,1


In [6]:
filename =  pathlib.Path(filepath).stem + "_cleaned.csv"
file_dest_path = pathlib.Path("/kaggle/working/") / filename
df.to_csv(file_dest_path, index=False)

In [7]:
#Droping the name column
tst_df.drop(['Name'], axis=1, inplace=True)

#building the deck and port features from the 'cabin' feature
tst_df['Deck'] = tst_df['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'M')
tst_df['Port'] = tst_df['Cabin'].apply(lambda s: s[-1] if pd.notnull(s) else 'M')
tst_df['Deck'] = tst_df["Deck"].map({'B':0, 'F':1, 'A':2, 'G':3, 'E':4, 'D':5, 'C':6, 'T':7, 'M':8})
tst_df['Port'] = tst_df["Port"].map({'P':0, 'S':1}).astype(int)
tst_df.drop(['Cabin'], axis=1, inplace=True)


tst_df['HomePlanet']= tst_df['HomePlanet'].map({'Earth':0, 'Europa':1,'Mars':2}).astype(int)

uni_destinations = tst_df["Destination"].unique()
tst_df["Destination"] = tst_df["Destination"].map(dict(zip(uni_destinations, list(range(len(uni_destinations)))))).astype(int)
tst_df["Destination"].unique()

if 'test' in tst_path:
    tst_df.drop(['PassengerId'],axis=1,inplace=True)

uni_cryosleep=tst_df["CryoSleep"].unique()
tst_df["CryoSleep"]=tst_df["CryoSleep"].map(dict(zip(uni_cryosleep, list(range(len(uni_cryosleep)))))).astype(int)

uni_VIP=tst_df["VIP"].unique()
tst_df["VIP"]=tst_df["VIP"].map(dict(zip(uni_VIP, list(range(len(uni_VIP)))))).astype(int)

tst_df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Deck,Port
0,0,0,0,27.0,0,0.0,0.0,0.0,0.0,0.0,3,1
1,0,1,0,19.0,0,0.0,9.0,0.0,2823.0,0.0,1,1
2,1,0,1,31.0,0,0.0,0.0,0.0,0.0,0.0,6,1
3,1,1,0,38.0,0,0.0,6652.0,0.0,181.0,585.0,6,1
4,0,1,0,20.0,0,10.0,0.0,635.0,0.0,0.0,1,1


In [8]:
filenames =  pathlib.Path(tst_path).stem + "_cleaned.csv"
file_dest_paths = pathlib.Path("/kaggle/working/") / filenames
tst_df.to_csv(file_dest_paths, index=False)

# Testing Model

In [9]:
train=pd.read_csv('/kaggle/working/train_cleaned.csv')

x=train.drop("Transported", axis=1).values
y=train["Transported"].values

In [10]:
test=pd.read_csv('/kaggle/working/test_cleaned.csv')

x=train.drop("Transported", axis=1).values
y=train["Transported"].values


In [11]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=40)
x_train.shape,y_train.shape,x_test.shape,y_test.shape

((6954, 12), (6954,), (1739, 12), (1739,))

In [12]:
gb=GradientBoostingClassifier()
gb.fit(x_train,y_train)
gb_pred_score=gb.score(x_test,y_test)   

In [13]:
df=pd.DataFrame(dict(model=[ 'Gradient Boosting'],
                     accuracy=[gb_pred_score]))

df

,model,accuracy
0,Gradient Boosting,0.797585


In [14]:
test=pd.read_csv('/kaggle/working/test_cleaned.csv')

tessst=pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

In [15]:
y_pred = gb.predict(test)
y_pred_bool = y_pred == 1

# 3. Create submission dataframe
submission = pd.DataFrame({
    "PassengerId":tessst["PassengerId"],
    "Transported": y_pred_bool
})

# 4. Save it
submission.to_csv("submission.csv", index=False)
submission=pd.read_csv('submission.csv')
submission

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GradientBoostingClassifier was fitted without feature names
  warnings.warn(


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
...,...,...
4272,9266_02,True
4273,9269_01,True
4274,9271_01,True
4275,9273_01,True
